# ECON 8876 SP 24 Session 8 (Lab)
## Instructions
Please do the following for this lab. Don't forget to upload your .ipynb file to CarmenCanvas Assignments.

In this lab session, we will go through the code of the Rust (1987) model.


# Preliminaries

Study the code below. Change the path to the location of busdata1234.csv in your google drive.

The zurcher class contains methods/functions that:


1.   Creates our grid for mileage (grid)
2.   Creates sparse transition matrices conditional on keep (P1) or replace (P2)
3.   Computes the Bellman function (ev1) and its derivative (dev1), and the choice probability conditional on keep (pk), given guess of value function (input ev0)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import time
import pandas as pd
import scipy.optimize as optimize
import matplotlib.pyplot as plt


path = '/content/drive/MyDrive/Session_8/'
os.chdir(path)
!pwd


class zurcher():
    def __init__(self,**kwargs):
        self.setup(**kwargs)

    def setup(self,**kwargs):
        # Code parameters
        self.n = 175 # Number of grid points
        self.max = 450 # Max mileage

        # Structural parameters: Default parameters (for Grps 1,2,3) as in Table X of Rust (1987)
        self.p = np.array([0.0937, 0.4475, 0.04459, 0.0127]) # Transition probability (theta_30,theta_31,theta_32,theta_33)
        self.RC = 11.7257 # Replacement cost
        self.c = 2.45569 # Cost parameter (theta_11)
        self.beta = 0.9999

        #If kwargs exist: Update the default values with the input values otherwise use the default parameters
        for key,val in kwargs.items():
            setattr(self,key,val)

        self.create_grid()

    def create_grid(self):
        self.grid = np.arange(0,self.n) # Mileage grid
        self.cost = 0.001*self.c*self.grid # Cost function (linear) scaled to $000
        self.state_transition()

    def state_transition(self):
        '''Compute transition probability matrixes conditional on choice'''
        p = np.append(self.p,1-np.sum(self.p)) # Get transition probabilities
        P1 = np.zeros((self.n,self.n)) # Initialize transition matrix
        # Loop over rows
        for i in range(self.n):
            # Check if p vector fits entirely
            if i <= self.n-len(p):
                P1[i][i:i+len(p)]=p
            else:
                P1[i][i:] = p[:self.n-len(p)-i]
                P1[i][-1] = 1.0-P1[i][:-1].sum()

        # conditional on d=1, replacement
        P2 = np.zeros((self.n,self.n))
        # Loop over rows
        for i in range(self.n):
            P2[i][:len(p)]=p
        self.P1 = P1
        self.P2 = P2

    def bellman(self,ev0,output=1):
        '''Evaluate Bellman operator, choice probability and Frechet derivative - written in integrated value form'''

        # Value of options:
        value_keep = -self.cost + self.beta * self.P1 @ ev0 # nx1 matrix
        value_replace = -self.RC - self.cost[0] + self.beta * self.P2 @ ev0   # 1x1

        # recenter Bellman by subtracting max(VK, VR)
        maxV = np.maximum(value_keep, value_replace)
        logsum = (maxV + np.log(np.exp(value_keep-maxV)  +  np.exp(value_replace-maxV)))  # Compute logsum to handle expectation over unobserved states
        ev1 = logsum # Bellman operator as integrated value instead of expected value Q. See discussion below regarding Newton-Kantrovich iterations

        if output == 1:
            return ev1

        # Compute choice probability of keep
        pk = 1/(1+np.exp(value_replace-value_keep))

        if output == 2:
            return ev1, pk

        # Compute derivative of Bellman operator
        dev1 = self.dbellman(pk)

        return ev1, pk, dev1

    def dbellman(self,pk):
        '''Compute derivative of Bellman operator'''
        dev1 = np.zeros((self.n,self.n))
        for d in range(2): # Loop over choices
            if d == 0:
                P = self.P1
                choice_prob =  pk
            else:
                P = self.P2
                choice_prob = 1-pk
        #
            dev1 += self.beta * choice_prob.reshape(-1, 1) * P
        return dev1


Print the following objects:


1.   Grid for mileage
2.   Transition probabilities P1 and P2
3.   Setting ev0 = np.zeros((model.n)), output the value function from one iteration of the Bellman operator.



In [ ]:
model = zurcher()

# Inner loop of NFXP

The inner loop of the Nested Fixed Point Algorithm solves for the fixed point of the Bellman equation. Rust (1987) actually uses the following "polyalgorithm" instead of just straight up using VFI:



*   Start with VFI.
*   If VFI convergence based on set tolerance, then done.
*   If switching is on and VFI iterations are very close to each other, then switch to Newton-Kantrovich iterations:

$$
Q_{k+1} = Q_{k} - (I-\Gamma'Q_k)^{-1} (Q_k-\Gamma Q_k)
$$

  where $\Gamma$ is the operator wrt expected value function $Q$ and $ \Gamma'Q_k$ is its derivative (dev1).


* End once either VFI or N-K converges.


It turns out the code converges much faster when using the Integrated value function instead of $Q$ in the NK iterations. The integrated value function is defined as

$$\overline{V}(x) = \int_\Omega V(x,\epsilon)q(\epsilon|x).$$

In this case, the Bellman equation in Integrated Value space becomes

$$
\overline{V} (x) = \int_\Omega \max_i \left[u(x,i) + \epsilon(i) + \beta \int_X \overline{V}(x')p(x'|x,i)dx'\right]q(\epsilon|x)d\epsilon
$$

Using Assumption EV, this gives

$$
\overline{V} (x) = \log \left[\sum_{i \in\{0,1\}} \left(u(x,i) + \beta \int_X \overline{V}(x')p(x'|x,i)dx'\right)\right] = \overline{\Gamma}\overline{V}
$$

We then define the NK iterations as
$$
\overline{V}_{k+1} = \overline{V}_{k} - (I-\overline{\Gamma}'\overline{V}_k)^{-1} (\overline{V}_k-\overline{\Gamma} \overline{V}_k)
$$


Study the code below.

In [7]:
class solve_NFXP():
    def __init__(self,**kwargs):
        self.setup(**kwargs)

    def setup(self,**kwargs):
        """" Set up the default values for the NFXP algorithm"""

        # VFI step
        self.vfi_max = 50      # Maximum number of contractions steps
        self.vfi_min = 10      # Minimum number of contraction steps
        self.vfi_tol = 1.0e-10 # Absolute tolerance before switching to Newton-Kantorovich iterations

        # Newton-Kantorovich steps
        self.max_fxpiter = 10     # Maximum number of times to switch between Newton-Kantorovich iterations and contraction iterations.
        self.pi_max = 10         # Maximum number of Newton-Kantorovich steps

        self.pi_tol = 1.0e-12    # Final exit tolerance in fixed point algorithm, measured in units of numerical precision
        self.tol_ratio = 1.0e-02 # Relative tolerance before switching to N-K algorithm when discount factor is supplied as input in solve.poly
        self.printfxp = 0        # Print iteration (0= no output), (1=compressed output), (2= detailed iteraion output)

        #If kwargs exist: Update the default values with the input values otherwise use the default settings
        for key,val in kwargs.items():
            setattr(self,key,val)

    def poly(self,bellman, V0=np.zeros(1), beta= 0.0, output=1):
        """" Solves the model using the poly-algorithm.
        set beta = 0.0 if you want to solve only with VFI.
        """

        t0poly = time.time()  # set the starting time

        # Loop over the maximum number of switches between contraction iterations and Newton-Kantorovich iterations
        for k in range(self.max_fxpiter):

            # 1. CONTRACTION ITERATIONS (VFI)
            if self.printfxp>0:
                print(f'Begin contraction iterations (for the {k+1} time)')
            V0,iter_vfi= self.vfi(bellman,V0,beta)

            # 2. NEWTON-KANTOROVICH ITERATIONS
            if self.printfxp>0:
                print(f'Begin Newton-Kantorovich iterations (for the {k+1} time)')
            V0,pk,dV, iter_nk = self.nk(bellman,V0)


            t1poly = time.time()
            if iter_nk.converged=='true':
                if self.printfxp>0:
                    print(f'Convergence achieved!')
                    print(f'Elapsed time: {(t1poly-t0poly):.4f} (seconds)')
                    break
            else:
                if k >= self.max_fxpiter:
                    print(f'No convergence! Maximum number of iterations exceeded without convergence!')
                    break
        V = V0
        if output==1:
            return V
        if output==2:
            return V, pk
        if output==3:
            return V, pk, dV
        if output==5:
            return V, pk, dV, iter_vfi, iter_nk
        else:
            print('solve_NFXP.poly: output must be 1,2,3 or 5')

    def vfi(self,bellman,V0=np.zeros(1), beta=0.0):
        """"Solves the model using the VFI algorithm.
        set beta = 0.0 if you want to solve only with VFI.
        beta = 0 means relative tolerance criterion used for switching is never met
        Default is just VFI when calling poly unless beta is set to >0.

        """
        #Empty class to store the iteration output
        class iteration: pass
        t0 = time.time()  # set the starting time
        iteration.tol = np.nan+np.zeros((self.vfi_max))
        iteration.rtol = np.nan+np.zeros((self.vfi_max))
        iteration.converged = 'false'
        # Loop over the maximum number of contraction iterations
        for i in range(self.vfi_max):
            V = bellman(V0,output=1) # Do contraction step
            iteration.tol[i] = max(abs(V-V0)) # Diff
            V0 = V.copy() # Update V0

            # Stopping criteria 1: Check if absolute tolerance is below the specified tolerance
            adj  = np.ceil(np.log10(abs(max(V0))))
            ltol = self.vfi_tol*10**adj  # Adjust to tolerance to compare with Diff
            if (i>=self.vfi_min) and (iteration.tol[i]<ltol):
                iteration.message = "VFI converged after {} iterations, tolerance: {:.4g}".format(i,iteration.tol[i])
                iteration.converged = 'true'
                break

            # Stopping criteria 2: Check the relative tolerance (ratio between current and previous Diff) and switch to N-K algorithm.
            # Idea is that once VFI is not jumping around (diff between successive iterations are close), switch to N-K
            if i>=self.vfi_min:
                iteration.rtol[i] = iteration.tol[i]/iteration.tol[max(i-1,0)]
                if (abs(beta-iteration.rtol[i]) < self.tol_ratio):
                    iteration.message = 'VFI stopped prematurely due to relative tolerance. Start N-K iterations'
                    iteration.converged = 'halfway'
                    break

        # Store the iteration output
        iteration.n = i+1
        iteration.tol = iteration.tol[0:i+1]
        iteration.rtol = iteration.rtol[0:i+1]
        t1 = time.time()            # set the ending time
        iteration.time = t1-t0

        self.print_output(iteration)

        return V, iteration

    def nk(self,bellman, V0):
        """"Solves the model using the Newton-Kantorovich steps"""
        #Empty class to store the iteration output
        class iteration: pass
        t0 = time.time()
        iteration.tol =  np.nan+np.zeros((self.pi_max))
        iteration.rtol = np.nan+np.zeros((self.pi_max))
        iteration.converged = 'false'
        # Get the state space size
        m = V0.size
        # Loop over the maximum number of Newton-Kantorovich steps
        for i in range(self.pi_max):

            # NK-step
            V1, pk, dV = bellman(V0,output=3) # Get V1 = Gamma V0 and its derivative given V0
            F = np.eye(m)-dV # Identity matrix - dV
            V = V0 - np.linalg.inv(F) @ (V0 - V1)  # Do N-K step

            # do additional VFI iteration for stability and accurate measure of error bound
            V0 = bellman(V,output=1)

            # Tolerance
            iteration.tol[i]=max(abs(V-V0))
            iteration.rtol[i] = iteration.tol[i]/(iteration.tol[max(i-1,0)] + 1.0e-15)

            #Adjust
            adj  = np.ceil(np.log10(abs(max(V0))))
            ltol = self.pi_tol*10**adj  # Adjust final tolerance

            if iteration.tol[i] < ltol:
                #Convergence achieved
                iteration.message = "N-K converged after {} iterations, tolerance: {:.4g}".format(i+1,iteration.tol[i])
                iteration.converged = 'true'
                break
        # Store the iteration output
        iteration.n = i+1
        iteration.tol = iteration.tol[0:i+1]
        iteration.rtol = iteration.rtol[0:i+1]
        t1 = time.time()
        iteration.time = t1-t0

        self.print_output(iteration)

        return V, pk, dV, iteration

    def print_output(self,iteration):
        if self.printfxp>1:           #print detaled output
            for i,iter_tol in enumerate(iteration.tol):
                print(f'Iteration {i+1}, tol {iter_tol:10.4g}, tol(j)/tol(j-1) {iteration.rtol[i]:10.4g}')

        if self.printfxp>0:           # print final output
            if iteration.converged != 'false':
                print(f'{iteration.message}')
            else:
                print(f'Maximum number of iterations reached, tolerance: {iteration.tol[-1]:.4f}')
            print(f'Elapsed time {iteration.time:.4f} seconds')


Solve first just using VFI and then compare results using the polyalgorithm.

In [ ]:
do_settings_solver = {
    'vfi_min': 10,
    'vfi_max': 10000,
    'printfxp': 2
}

solver = solve_NFXP(**do_settings_solver)

ev0 = np.zeros((model.n)) # Initial guess

# VFI
print('\n\nRunning VFI only\n\n')

# Poly
print('\n\nRunning Rust\'s polyalgorithm\n\n')


# Read data

The following code reads and loads the data busdata1234.csv. Make sure you set the path at the beginning correctly to point to the folder where the file is located.

In [9]:
def read_busdata(model, bustypes = [1,2,3,4]):
        data = np.loadtxt(open("busdata1234.csv"), delimiter=",")
        idx = data[:,0]             # bus id
        bustype = data[:,1]         # bus type
        dl = data[:,4]              # laggend replacement dummy
        d = np.append(dl[1:], 0)    # replacement dummy
        x = data[:,6]               # Odometer

        # Discretize odometer data into 1,2,...,n
        x = np.ceil(x*model.n/(model.max*1000))

        # Montly mileage
        dx1 = x-np.append(0,x[0:-1])
        dx1 = dx1*(1-dl)+x*dl
        dx1 = np.where(dx1>len(model.p),len(model.p),dx1) # We limit the number of steps in mileage

        # change type to integrer
        x = x.astype(int)
        dx1 = dx1.astype(int)

        # Collect in a dataframe
        remove_first_row_index=idx-np.append(0,idx[:-1])
        data = {'id': idx,'bustype':bustype, 'd': d, 'x': x, 'dx1': dx1, 'boolean': remove_first_row_index}
        df= pd.DataFrame(data)

        # Remove observations with missing lagged mileage
        df = df.drop(df[df.boolean!=0].index)

        # Select bustypes
        for j in [1,2,3,4]:
            if j not in bustypes:
                df = df.drop(df[df.bustype==j].index)

        # save data
        dta = df.drop(['id','boolean'],axis=1)

        return dta

Call read_busdata and take a look at its output

In [ ]:
data =

# Estimation
The following code contains the methods to estimate the parameters of the model using the NFXP, i.e. MLE subject to $$\overline{V} = \overline{\Gamma}\overline{V}$$

Study the code below focusing on the function estimate() and ll().

In [16]:
ev = np.zeros(1) # Global variable

def estimate(model,solver,data,theta0=[0,0],twostep=0):
    """" Estimate model using NFXP"""
    global ev
    ev = np.zeros((model.n))

    samplesize = data.shape[0]

    # STEP 1: Find p non-parametrically
    tabulate = data.dx1.value_counts() # Count number of observations for each dx1
    p = [tabulate[i]/sum(tabulate) if i < len(tabulate) else 0 for i in range(len(model.p))]

    # STEP 2: Estimate structual parameters
    model.p[:] = p # Use first step estimates as starting values for p. When twostep = 0, we also estimate p using NFXP instead of nonparametrically

    ## Estimate RC and C
    pnames = ['RC','c']

    # Call BHHH optimizer
    res = optimize.minimize(ll,theta0,args = (model, solver, data, pnames), method = 'trust-ncg',jac = grad, hess = hes, tol=1e-8)
    # Update parameters
    model = updatepar(model,pnames,res.x)

    # Estimate RC, c and p
    if twostep == 0:
        pnames = ['RC','c','p']
        theta0 = [model.RC, model.c] + model.p.tolist() # Starting values
        # Call BHHH optimizer
        res = optimize.minimize(ll,theta0, args = (model,solver,data, pnames), method = 'trust-ncg',jac = grad, hess = hes, tol = 1e-8)

        # Update parameters
        model=updatepar(model,pnames,res.x)

    # Converged: "trust-ncg tends to be very conservative about convergence, and will often return status 2 even when the solution is good."
    converged   =   (res.status == 2 or res.status ==0)

    # Compute Variance-Covaiance matrix
    h = hes(res.x, model, solver,data, pnames) # Hessian
    Avar = np.linalg.inv(h*samplesize) # Variance-Covariance matrix from information matrix equality

    theta_hat = res.x # unpack estimates

    return model, res, pnames, theta_hat, Avar, converged

def ll(theta, model, solver,data, pnames, out=1, no_guess = False): # out=1 solve optimization
    """ Compute log-likelihood function """
    global ev # Use global variable to store value function to use as starting value for next iteration

    #Unpack and convert to numpy array
    x = np.array(data.x - 1) # x is the index of the observed state: We subtract 1 because python starts counting at 0
    d = np.array(data.d) # d is the observed decision
    dx1 = np.array(data.dx1) # dx1 is observed change in x

    if no_guess == True: #Set ev to zero instead of using global variable
        ev = np.zeros((model.n))

    # Update values
    model=updatepar(model,pnames,theta)
    model.p = np.abs(model.p)    # helps BHHH which is run as unconstrained optimization
    model.create_grid() # Update grid
    ev0 = ev # Use previous value function as starting value

    # Solve the model
    ev, pk, dev = solver.poly(model.bellman, V0=ev0 ,beta=model.beta, output=3)

    # Evaluate likelihood function
    lik_pr = pk[x] # Get probability of keeping given observed state
    choice_prob = lik_pr * (1 - d) + (1-lik_pr) * d # get probability of making observed choice
    log_lik = np.log(choice_prob)  # Compute log-likelihood-contributions

    # add on log like for mileage process
    if theta.size>2: # theta > 2 if there are parameters for p
        p = np.append(model.p,1-np.sum(model.p)) # Add residual probability to p
        if any(p<=0):
            log_lik -= 100000*p[dx1] # Penalize if p is negative
        else:
            log_lik += np.log(p[dx1]) # Add log-likelihood contribution

    else:
        p = np.nan


    if out == 1:
        # Objective function (negative mean log likleihood)
        return np.mean(-log_lik)

    return model,lik_pr, pk, ev, dev, d,x,dx1


def score(theta, model, solver, data, pnames):
    """ Compute score function """
    global ev # Use global variable to store value function to use as starting value for next iteration
    model,lik_pr, pk, ev, dev, d,x,dx1 = ll(theta, model, solver, data, pnames,9) # Evaluate likelihood function
    F = np.eye(model.n)-dev # Get frechet derivative
    N = data.x.size # Number of observations
    dc = 0.001*model.grid # Get derivative of cost function in utility wrt c
    pk = pk.reshape((model.n,1))  # Reshape to get correct shape for matrix multiplication



    ##### COMPUTE THE SCORE #######
    # Check if there are parameters for p
    if theta.size>2:
        n_p = len(model.p)
    else:
        n_p = 0


    # STEP 1: compute derivative of contraction operator wrt. parameters

    ## Derivative of utility function wrt. parameters
    dutil_dtheta=np.zeros((model.n, 2 + n_p, 2)) # shape is (gridsize, number of parameters, number of choices in utility function)
    dutil_dtheta[:,0, 0] = 0 # derivative of keeping wrt RC
    dutil_dtheta[:,0, 1] = -1 # derivative of replacing wrt RC
    dutil_dtheta[:,1, 0] = -dc # derivative of keeping wrt c
    dutil_dtheta[:,1, 1] = -dc[0] # derivative of replacing wrt c

    # Derivative of contraction operator wrt. utility parameters
    dbellman_dtheta=np.zeros((model.n, 2 + n_p)) # shape is (gridsize, number of parameters)
    dbellman_dtheta[:,:] =  (pk * dutil_dtheta[:, :, 0] + (1 - pk) * dutil_dtheta[:, :, 1])
    #dbellman_dtheta = model.P1@dbellman_dtheta # Since use Q instead of integrated value function


    # Derivative of contraction operator wrt. p
    if theta.size>2:
        vk = -model.cost + model.beta * model.P1 @ ev # Value of keeping
        vr = -model.RC-model.cost[0]+model.beta *model.P2 @ ev # Value of replacing
        vmax = np.maximum(vk,vr) # Get maximum value
        dbellman_dpi = vmax+np.log(np.exp(vk-vmax)+np.exp(vr-vmax)) #Re-centered log-sum: Value functin
        for i_p in range(n_p): # loop over p
            part1 = dbellman_dpi[i_p:-1]
            part2 = np.hstack((dbellman_dpi[n_p:model.n], np.tile(dbellman_dpi[-1],(n_p-i_p-1))))
            dbellman_dtheta[0:model.n-i_p-1,2+i_p] =part1-part2

        invp=np.exp(-np.log(model.p))
        invp = np.vstack((np.diag(invp[0:n_p]),-np.ones((1,n_p))*invp[n_p-1]))

    # STEP 2: Compute derivative of Fixed point wrt. parameter
    dev_dtheta = np.linalg.solve(F,dbellman_dtheta)

    # STEP 3: Compute derivative of log-likelihood wrt. parameters


    score = np.zeros((N, 2 + n_p)) # Initialize score function
    for d_loop in range(2): # Loop over decisions (keep=0, replace=1)
        # Get transition matrix
        if d_loop == 0:
            P = model.P1
        else:
            P = model.P2
        dv = dutil_dtheta[:, :, d_loop] + model.beta * P @ dev_dtheta  # derivative of choice-specific value function wrt. parameters
        choice_prob = lik_pr * (1 - d_loop) + (1-lik_pr) * d_loop # get probability of choice in loop
        score += ((d == d_loop) -  choice_prob ).reshape(-1,1) * dv[x, :] # Add derivative of log-likelihood wrt. parameters

    # Add derivative of log-likelihood from mileage process wrt. p
    if theta.size>2:
        for i_p in range(n_p):
            score[:,2+i_p] = score[:,2+i_p]+invp[dx1,i_p]

    return score

def grad(theta, model, solver,data, pnames):
    """ Compute gradient of log-likelihood function """
    s = score(theta, model, solver, data,pnames)
    return -np.mean(s,0)

def hes(theta, model, solver,data, pnames):
    """ Compute Hessian of log-likelihood function as outer product of scores"""
    s= score(theta, model, solver, data, pnames)

    return s.T@s/data.shape[0]

def updatepar(par,parnames, parvals):
    """ Update parameters """
    for i,parname in enumerate(parnames):
        # First two parameters are scalars
        if i<2:
            parval = parvals[i]
            setattr(par,parname,parval)
        else: # Remaining parameters are lists
            list_val = [None]*(parvals.size-2)
            for j,parval in enumerate(parvals[2:]):
                list_val[j]=parval
            setattr(par,parname,list_val)
    return par


# Estimate the model
Esitmate the model with beta = 0.9999 and beta = 0. Compare our estimate of the parameter c.


In [ ]:
do_settings = {
    'beta' :
}
model = zurcher(**do_settings)

do_settings_solver = {
    'printfxp': 0
}
solver = solve_NFXP(**do_settings_solver)



# Estimate the model
import time
t0 = time.time()
theta0 = [0,0]

# args for nfxp estimate
nfxp_model, optim_res, pnames, theta_hat, Avar, converged=estimate(model, solver,data,theta0=theta0, twostep=0)

t1 = time.time()
time = t1-t0

# Print the result
print(f'Structual estimation using busdata from Rust(1987)')
print(f'Beta        = {model.beta:.4f}')
print(f'n           = {model.n}')
print(f'Sample size = {samplesize}\n \n')

print(f'Parameters     Estimates    s.e. ')
print(f'{pnames[0]}             {theta_hat[0]:.4f}     {np.sqrt(Avar[0,0]):.4f} ')
print(f'{pnames[1]}              {theta_hat[1]:.4f}     {np.sqrt(Avar[1,1]):.4f} \n ')
print(f'{pnames[2]}(1)           {theta_hat[2]:.4f}     {np.sqrt(Avar[2,2]):.4f}  ')
print(f'{pnames[2]}(2)           {theta_hat[3]:.4f}     {np.sqrt(Avar[3,3]):.4f}  ')
print(f'{pnames[2]}(3)           {theta_hat[4]:.4f}     {np.sqrt(Avar[4,4]):.4f}  ')
print(f'{pnames[2]}(4)           {theta_hat[5]:.4f}     {np.sqrt(Avar[5,5]):.4f}  \n')


print(f'Log-likelihood {-optim_res.fun*samplesize:.2f}')
print(f'runtime (seconds) {time:.4f}')
print(f'The model converged: {converged}')